# Notebook 5: Savunma Mekanizmalari

In [1]:
import sys
sys.path.append('..')
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import accuracy_score, f1_score
from src.defense import build_defended_models
from src.utils import load_model, save_model, print_section, FIGURES_DIR
print('Import basarili')

Import basarili


In [2]:
X_train = np.load('../data/processed/X_train.npy')
X_test  = np.load('../data/processed/X_test.npy')
y_train = np.load('../data/processed/y_train.npy')
y_test  = np.load('../data/processed/y_test.npy')
X_adv_combined = np.load('../data/processed/X_adv_combined.npy')
with open('../data/processed/feature_names.json') as f:
    feature_names = json.load(f)
baseline_models = {
    'Random Forest':       load_model('baseline_random_forest'),
    'XGBoost':             load_model('baseline_xgboost'),
    'Logistic Regression': load_model('baseline_logistic_regression'),
}
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

[2026-02-28 12:38:26] INFO [utils] Model yüklendi: C:\Users\emira\OneDrive\Desktop\main projem\notebooks\..\data\models\baseline_random_forest.pkl
[2026-02-28 12:38:26] INFO [utils] Model yüklendi: C:\Users\emira\OneDrive\Desktop\main projem\notebooks\..\data\models\baseline_xgboost.pkl
[2026-02-28 12:38:26] INFO [utils] Model yüklendi: C:\Users\emira\OneDrive\Desktop\main projem\notebooks\..\data\models\baseline_logistic_regression.pkl


Train: (24987, 64) | Test: (25013, 64)


In [3]:
print_section('Robust Model Egitimi')
defended_models = build_defended_models(baseline_models, X_train, y_train, feature_names)
print('Savunmali modeller:', list(defended_models.keys()))


════════════════════════════════════════════════════════════
  Robust Model Egitimi
════════════════════════════════════════════════════════════


  SAVUNMA MODELLERİ EĞİTİLİYOR

🛡️  Random Forest için robust model eğitiliyor...

🛡️  RobustClassifier eğitiliyor...
   Smoothing: True
   Adversarial Training: True
✅ Feature Smoothing fit edildi (percentile)
🛡️  Adversarial Training başlıyor...
   Base model: RandomForestClassifier
   Adversarial ratio: 0.25
   Augmentation rounds: 3
   Augmented dataset boyutu: 43725 (orijinal: 24987)
✅ Adversarial training tamamlandı
✅ RobustClassifier hazır

🛡️  XGBoost için robust model eğitiliyor...

🛡️  RobustClassifier eğitiliyor...
   Smoothing: True
   Adversarial Training: True
✅ Feature Smoothing fit edildi (percentile)
🛡️  Adversarial Training başlıyor...
   Base model: XGBClassifier
   Adversarial ratio: 0.25
   Augmentation rounds: 3
   Augmented dataset boyutu: 43725 (orijinal: 24987)
✅ Adversarial training tamamlandı
✅ RobustClassifier ha

In [4]:
rows = []
all_models = {**baseline_models, **defended_models}
for model_name, model in all_models.items():
    acc_normal = accuracy_score(y_test, model.predict(X_test))
    acc_adv    = accuracy_score(y_test, model.predict(X_adv_combined))
    rows.append({'Model': model_name, 'Normal Acc': round(acc_normal,4), 'Adversarial Acc': round(acc_adv,4), 'Drop': round(acc_normal-acc_adv,4)})
result_df = pd.DataFrame(rows).set_index('Model')
print(result_df.to_string())

                            Normal Acc  Adversarial Acc    Drop
Model                                                          
Random Forest                   0.9271           0.9038  0.0233
XGBoost                         0.9392           0.9409 -0.0017
Logistic Regression             0.8900           0.7830  0.1071
Random Forest_robust            0.9101           0.9777 -0.0675
XGBoost_robust                  0.9351           0.9797 -0.0445
Logistic Regression_robust      0.8632           0.8882 -0.0251
ensemble                        0.9322           0.9068  0.0254


In [5]:
fig, ax = plt.subplots(figsize=(12,6))
x = np.arange(len(result_df))
ax.bar(x-0.175, result_df['Normal Acc'], 0.35, label='Normal', color='#4CAF50', alpha=0.8)
ax.bar(x+0.175, result_df['Adversarial Acc'], 0.35, label='Adversarial', color='#F44336', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(result_df.index, rotation=15, fontsize=9)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Accuracy')
ax.set_title('Savunma Oncesi vs Sonrasi')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'defense_comparison.png', dpi=150, bbox_inches='tight')
print('Grafik kaydedildi')

Grafik kaydedildi


In [6]:
for name, model in defended_models.items():
    safe_name = name.lower().replace(' ','_').replace('(','').replace(')','').replace('/','_')
    save_model(model, f'defended_{safe_name}')
result_df.to_csv('../data/processed/final_summary.csv')
print('Notebook 5 tamamlandi!')
print('Sonraki adim: 6_explainability_shap.ipynb')

Model kaydedildi: C:\Users\emira\OneDrive\Desktop\main projem\data\models\defended_random_forest_robust.pkl
Model kaydedildi: C:\Users\emira\OneDrive\Desktop\main projem\data\models\defended_xgboost_robust.pkl
Model kaydedildi: C:\Users\emira\OneDrive\Desktop\main projem\data\models\defended_logistic_regression_robust.pkl
Model kaydedildi: C:\Users\emira\OneDrive\Desktop\main projem\data\models\defended_ensemble.pkl
Notebook 5 tamamlandi!
Sonraki adim: 6_explainability_shap.ipynb
